In [50]:
import os
import time
import random
import warnings
from datetime import date

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree # type: ignore <- pylance milně hlásí chybu
from pathlib import Path
import time
import sys
import polars as pl
import polars.selectors as cs
import json
import pickle
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patheffects as path_effects
from ydata_profiling import ProfileReport
import geopandas as gpd


current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
sys.path.append(str(current_dir.parent))
from utils import *
from schemas import *
from clean import *
from visualisation_utils import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')
# os.chdir(r'C:\Users\adamp\Projects\CVUT_BAP')
SEED=42
PRINT = False

# Načtení dat

In [51]:
df = pl.read_parquet(r'kod\data\extracted\prohlidky_vozidel_stk_a_sme\parquet', schema=prohlidky_schema)
if PRINT: describe(df)

# Filtrace vozidel kategorie M1 s palivem Benzín / Nafta

In [52]:
if PRINT:
    display(df['Vozidlo_Kategorie'].value_counts(sort=True))
    display(df['Emise_ZakladniPalivo'].value_counts(sort=True))
    display(df['Emise_AlternativniPalivo'].value_counts(sort=True))

In [53]:
df = df.filter(pl.col('Vozidlo_Kategorie') == 'M1', pl.col('Emise_ZakladniPalivo').is_in(['Benzín', 'Nafta']), pl.col('Emise_AlternativniPalivo').is_null())
if PRINT: schema_description(df)

# Přetypování sloupců
Velke mnozstvi zaznamu ma odlisne cislo protokolu pouze z duvodu predchazejicich nul

In [54]:
if PRINT:
    print('Záznamy, kde neodpovídá CisloProtokolu a Emise_CisloProtokolu')
    short_display(df[['CisloProtokolu', 'Emise_CisloProtokolu']].filter(pl.col('CisloProtokolu') != pl.col('Emise_CisloProtokolu')))
df = cast_prohlidka(df)

# Odstranění nekonzistentních CiselProtokolu
Pouze 3 cisla protokolu nejsou stejna

In [55]:
inconsistent_cislo_protokolu = df[['CisloProtokolu', 'Emise_CisloProtokolu']].filter(pl.col('CisloProtokolu') != pl.col('Emise_CisloProtokolu'))['CisloProtokolu'].to_list()
if PRINT: print(inconsistent_cislo_protokolu)
df = df.filter(~pl.col('CisloProtokolu').is_in(inconsistent_cislo_protokolu))

# Odstranění duplicitních čísel protokolu
Nektere zaznamy se v datove sade vyskytuji vicektrat

In [56]:
if PRINT: short_display(df.filter(pl.col('CisloProtokolu').count().over('CisloProtokolu') > 1).sort('CisloProtokolu'))
df = df.unique()
if PRINT: short_display(df.filter(pl.col('CisloProtokolu').count().over('CisloProtokolu') > 1).sort('CisloProtokolu'))

# Zapracování administrativních oprav
## Charakteristiky administrativnich oprav
- Nektere protokoly se jsou opravovany vicekrat
- Nektere administrativni opravy se odkazuji na protokoly, ktere v datove sade nejsou

In [57]:
df_repairs = df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null())
if PRINT: print(f'Pocet administrativnich oprav: {len(df_repairs)}')

if PRINT: print('Pocty oprav alespon jednou opravovanych protokolu:')
if PRINT: display(df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null())['AdministrativniOprava_CisloProtokolu'].value_counts().group_by(by='count').len().sort(by='by'))

df_repairs_of_repairs = df.filter(pl.col("AdministrativniOprava_CisloProtokolu").is_in(df.filter(pl.col("AdministrativniOprava_CisloProtokolu").is_not_null())["CisloProtokolu"].to_list()))
if PRINT: print(f'Pocet oprav opravujicich opravy: {len(df_repairs_of_repairs)}')


### Prevedeni odkazu administrativni opravy na korenovy zaznam

In [58]:
parent_map = dict(zip(df["CisloProtokolu"], df["AdministrativniOprava_CisloProtokolu"]))

def get_absolute_root(protocol_id):
    current = protocol_id
    visited = set()
    while parent_map.get(current) is not None:
        if current in visited:
            raise ValueError("Cyklicka reference")
        visited.add(current)
        current = parent_map.get(current)
    return current

df = df.with_columns(
    pl.col("CisloProtokolu").map_elements(get_absolute_root, return_dtype=pl.Utf8).alias("Absolutni_Koren")
).with_columns(
    pl.when(pl.col("CisloProtokolu") != pl.col("Absolutni_Koren"))
    .then(pl.col("Absolutni_Koren"))
    .otherwise(pl.lit(None))
    .alias("AdministrativniOprava_CisloProtokolu")
).drop("Absolutni_Koren")

### Odstraneni sirotcich oprav

In [59]:
df_orphaned_repairs = df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null()).join(df.select('CisloProtokolu'), left_on='AdministrativniOprava_CisloProtokolu', right_on='CisloProtokolu', how='anti')
if PRINT: print(f'Pocet administrativnich oprav, ktere se odkazuji na neexistujici protokol: {len(df_orphaned_repairs)}')
df = df.join(df_orphaned_repairs.select("CisloProtokolu"), on="CisloProtokolu", how="anti")

### Situace po sjednoceni oprav

In [60]:
df_repairs = df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null())
if PRINT: print(f'Pocet administrativnich oprav: {len(df_repairs)}')

if PRINT: print('Pocty oprav alespon jednou opravovanych protokolu:')
if PRINT: display(df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null())['AdministrativniOprava_CisloProtokolu'].value_counts().group_by(by='count').len().sort(by='by'))

df_repairs_of_repairs = df.filter(pl.col("AdministrativniOprava_CisloProtokolu").is_in(df.filter(pl.col("AdministrativniOprava_CisloProtokolu").is_not_null())["CisloProtokolu"].to_list()))
if PRINT: print(f'Pocet oprav opravujicich opravy: {len(df_repairs_of_repairs)}')

df_orphaned_repairs = df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null()).join(df.select('CisloProtokolu'), left_on='AdministrativniOprava_CisloProtokolu', right_on='CisloProtokolu', how='anti')
if PRINT: print(f'Pocet administrativnich oprav, ktere se odkazuji na neexistujici protokol: {len(df_orphaned_repairs)}')

## Identifikace změn
Je potreba pocitat s tim, ze nekktere protokoly byly meneny vicekrat

In [61]:
# Definice ignorovanych
ignore_cols = {
    'DatumProhlidky',
    'Prohlidka_Zahajeni', 
    'Prohlidka_Ukonceni', 
    'AdministrativniOprava_CisloProtokolu', 
    'AdministrativniOprava_DatumProhlidky',
    'CisloProtokolu',
    'Emise_CisloProtokolu',
    'Emise_DatumProhlidky',
    'Emise_Zahajeni',
    'Emise_Ukonceni',
    'Vysledek_Poznamka',
}
    
if PRINT:
    # Dynamické určení sledovaných sloupců
    compare_cols = [c for c in df.columns if c not in ignore_cols]

    def to_str_expr(col_name):
        dtype = df.schema[col_name]
        if isinstance(dtype, pl.List):
            return pl.col(col_name).list.join(", ").cast(pl.Utf8)
        return pl.col(col_name).cast(pl.Utf8)

    # Finální optimalizovaný proces
    final_diff_table = (
        df.lazy()
        .with_columns(
            pl.coalesce("AdministrativniOprava_CisloProtokolu", "CisloProtokolu").alias("Family_ID")
        )
        .sort("Prohlidka_Ukonceni")
        .select([
            pl.col("CisloProtokolu").alias("Protokol_Opravy"),
            pl.col("Family_ID").alias("Opravovany_Protokol"),
            *[
                pl.when(
                    (pl.col(c) != pl.col(c).shift().over("Family_ID")) |
                    (pl.col(c).is_null() != pl.col(c).shift().over("Family_ID").is_null())
                )
                .then(
                    pl.struct(
                        Zmeneny_Parametr=pl.lit(c),
                        Puvodni_Hodnota=to_str_expr(c).shift().over("Family_ID"),
                        Nova_Hodnota=to_str_expr(c)
                    )
                )
                .otherwise(None)
                .alias(c)
                for c in compare_cols
            ]
        ])
        # Odstranění řádků, které nejsou opravami (kořenové protokoly)
        .filter(pl.col("Protokol_Opravy") != pl.col("Opravovany_Protokol"))
        .unpivot(
            index=["Protokol_Opravy", "Opravovany_Protokol"],
            on=compare_cols,
            value_name="Changes"
        )
        .filter(pl.col("Changes").is_not_null())
        .unnest("Changes")
        .drop("variable") # unpivot automaticky vytváří 'variable', zde duplicitní k Zmeneny_Parametr
        .sort(["Opravovany_Protokol", "Protokol_Opravy"])
        .collect()
    )

In [62]:
if PRINT: print(f'Prumerny pocet menenych parametru: {len(final_diff_table)/len(df_repairs)}')

## Vykreslení nejčastěji měněných parametru

In [63]:
if PRINT:
    # Agregace počtu změn na parametr
    summary = final_diff_table.group_by('Zmeneny_Parametr').len().sort('len', descending=True)

    columns = summary['Zmeneny_Parametr'].to_list()
    counts = summary['len'].to_list()
    total_revisions = df_repairs.height
    ratios = [c / total_revisions if total_revisions > 0 else 0 for c in counts]

    group_indices = [prohlidky_group_mapping[col] for col in columns]
    group_descriptions = ['Identifikační údaje', 'Stanice', 'Časové údaje', 'Vozidlo', 'Registrace', 'Emise', 'Jiný typ kontroly', 'Výsledek', 'Administrativní oprava']
    horizontal_bar(columns, ratios, 'Četnost změn v jednotlivých parametrech (Administrativní opravy)', "kod/explorace/prohlidky_grafy/prohlidky_administrativni_opravy.svg", 3, 10, group_indices, group_descriptions)

## Zanesení administrativních oprav
Nasledujici kod funguje diky uprave na korenovy protokol

In [64]:
# Sloupce určené k přepsání daty z oprav
update_cols = [c for c in df.columns if c not in ignore_cols]

# Získání posledních verzí oprav pro každý protokol
latest_corrections = (
    df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_not_null())
    .sort('Prohlidka_Ukonceni')
    .group_by('AdministrativniOprava_CisloProtokolu', maintain_order=True)
    .last()
    .with_columns(pl.lit(True).alias('_has_correction'))
)

# Aplikace oprav na originální záznamy
df = (
    df.filter(pl.col('AdministrativniOprava_CisloProtokolu').is_null())
    .join(
        latest_corrections.select(
            ['AdministrativniOprava_CisloProtokolu', '_has_correction'] + 
            [pl.col(c).alias(f"{c}_Latest") for c in update_cols]
        ),
        left_on='CisloProtokolu',
        right_on='AdministrativniOprava_CisloProtokolu',
        how='left'
    )
    .with_columns([
        # Nahrazení hodnotou z opravy, pokud existuje (včetně null hodnot)
        *[
            pl.when(pl.col('_has_correction').fill_null(False))
            .then(pl.col(f"{c}_Latest"))
            .otherwise(pl.col(c))
            .alias(c)
            for c in update_cols
        ],
        # Informace o tom, zda byl zaznam opravovan
        pl.col('_has_correction').fill_null(False).alias('AdministrativneOpraveno'),
    ])
    .select([*df.columns, "AdministrativneOpraveno"]) # Vezme vše původní + nový bool
    .select(pl.exclude(['AdministrativniOprava_CisloProtokolu', 'AdministrativniOprava_DatumProhlidky']))
)

# Odstranění vybraných sloupců
- nerelevantních sloupců
- sloupců přítomných v jiných datových sadách s vyšší kvalitou
- sloupcu pritomnych je u techncikych prohlidek


In [ ]:
irrelevant_columns = ['Registrace_CisloDokladu', 'Vozidlo_Vin', 'Vozidlo_Kategorie', 'Emise_CisloProtokolu', 'Emise_AlternativniPalivo', 'Emise_VyrobceMotoru', 'Emise_DatumProhlidky', 'Emise_CisloMotoru', 'Vysledek_Poznamka', 'Vysledek_DatumPristiProhlidky', 'Vysledek_Celkovy', 'Zavady_A', 'Zavady_B', 'Zavady_C']
duplicate_columns = ['Emise_EmisniSystem']
df = df.drop(irrelevant_columns + duplicate_columns)

In [66]:
if PRINT:
    print(f'Pocet zanamu obsahujicich RozsahProhlidky bez TechnickeCasti_Pritomno: {df.filter(pl.col('RozsahProhlidky').is_not_null()).select((pl.col('TechnickaCast_Pritomno') == False).sum()).item()}.')
    print(f'Pocet zanamu obsahujicich Vysledek_NalepkaVylepena bez TechnickeCasti_Pritomno: {df.filter(pl.col('Vysledek_NalepkaVylepena').is_not_null()).select((pl.col('TechnickaCast_Pritomno') == False).sum()).item()}.')

df = df.drop('RozsahProhlidky')
df = df.drop('Vysledek_NalepkaVylepena')

In [67]:
if PRINT: print(f'Pocet stanic, kde se nerovna cislo stanice prohlidky a emise: {df.filter(pl.col('Prohlidka_Stanice_Cislo') != pl.col('Emise_StaniceCislo')).height}')
df = df.drop('Emise_StaniceCislo')

## Dataset po zanesení administrativních kontrol

In [68]:
df = df.select(['CisloProtokolu', 'DruhProhlidky', 'Prohlidka_OdpovednaOsoba', 'Prohlidka_Stanice_Cislo', 'Prohlidka_Stanice_Kraj', 'Prohlidka_Stanice_ORP', 'Prohlidka_Stanice_Obec', 'DatumProhlidky', 'Prohlidka_Zahajeni', 'Prohlidka_Ukonceni', 'Vozidlo_Vin', 'Vozidlo_Druh', 'Vozidlo_Provedeni', 'Vozidlo_Znacka', 'Vozidlo_ObchodniOznaceni', 'Vozidlo_TypMotoru', 'Registrace_DatumPrvni', 'Registrace_Stat', 'Emise_OdpovednaOsoba', 'Emise_Zahajeni', 'Emise_Ukonceni', 'Emise_ZakladniPalivo', 'TechnickaCast_Pritomno', 'AdrCast_Pritomno', 'TskCast_Pritomno', 'Vysledek_Odometr', 'AdministrativneOpraveno'])
if PRINT: schema_description(df)

# Chybějící hodnoty

In [69]:
if PRINT:
    # Výpočet metrik
    total_rows = len(df)
    columns = df.columns
    null_counts = df.null_count().row(0)
    non_null_counts = [total_rows - count for count in null_counts]
    ratios = [count / total_rows if total_rows > 0 else 0 for count in non_null_counts]

    prohlidky_group_mapping['AdministrativneOpraveno'] = 8
    group_indices = [prohlidky_group_mapping[col] for col in columns]

    group_descriptions = ['Identifikační údaje', 'Stanice', 'Časové údaje', 'Vozidlo', 'Registrace', 'Emise', 'Jiný typ kontroly', 'Výsledek', 'Administrativní oprava']
    horizontal_bar(
        labels=columns, 
        counts=ratios, 
        title='Počty přítomných hodnot v datové sérii prohlídek', 
        save_path='kod/explorace/prohlidky_grafy/prohlidky_pritomne_hodnoty.svg', 
        decimals=3, 
        height=14, 
        group_indices=group_indices, 
        group_descriptions=group_descriptions
    )
    print(f'Pocet chybejicich hodnot u "Registrace_DatumPrvni": {df['Registrace_DatumPrvni'].null_count()}')

# Analýza jednotlivých příznaků
## Druh prohlídky
- Odstraneni tohoto sloupce

In [70]:
if PRINT:
    exam_types = df['DruhProhlidky'].value_counts().sort(by='count', descending=True)
    horizontal_bar(exam_types['DruhProhlidky'], exam_types['count'] / df.height, 'Hodnoty v atributu "DruhProhlidky"', 'kod/explorace/prohlidky_grafy/prohlidky_druh_prohlidky.svg', max_bars=4, height=4, decimals=3)
    print(f'Podíl opakovaných prohlídek na všech {exam_types.filter(pl.col('DruhProhlidky') == 'Opakovaná')['count'].item() / exam_types['count'].sum() * 100:.3} %')
df = df.drop('DruhProhlidky')

## Prohlidka, Emise odpovedna osoba

In [71]:
if PRINT:
    print(df['Prohlidka_OdpovednaOsoba'].n_unique())
    df.group_by("Prohlidka_OdpovednaOsoba").agg(pl.col("Prohlidka_Stanice_Cislo").n_unique().alias('PocetStanic')).filter(pl.col('PocetStanic') > 1).sort(by='PocetStanic', descending=True)

In [72]:
if PRINT:
    print(f'Podil pripadu, kdy se odpovedna osoba v prohlidce a v emisisch neshoduje: {df.filter(pl.col('Prohlidka_OdpovednaOsoba') != pl.col('Emise_OdpovednaOsoba')).height / df.height *100:.2f} %')
    print(f'Podil pripadu, kdy se odpovedna osoba v prohlidce a v emisisch neshoduje (technicka pritomno): {df.filter(pl.col('TechnickaCast_Pritomno') == True).filter(pl.col('Prohlidka_OdpovednaOsoba') != pl.col('Emise_OdpovednaOsoba')).height / df.filter(pl.col('TechnickaCast_Pritomno') == True).height *100:.2f} %')

## Prohlidka, Emise stanice cislo

In [73]:
if PRINT:
    # Pomocne informace
    timeframe = df.select(diff_years = (pl.col("DatumProhlidky").max() - pl.col("DatumProhlidky").min()).dt.total_days() / 365.24).item()
    provozovatel_mapping = pl.read_parquet(r"E:\CVUT_BAP\kod\data\processed\stanice.parquet")

    # Samotne nalezeni top stanic
    top_stanice = df['Prohlidka_Stanice_Cislo'].value_counts().sort(by='count', descending=True).head(10)

    labels = top_stanice.select(pl.col('Prohlidka_Stanice_Cislo').cast(pl.String)).join(provozovatel_mapping, how='left', left_on='Prohlidka_Stanice_Cislo', right_on='Stanice_Cislo').select(merged = pl.col("Prohlidka_Stanice_Cislo") + " (" + pl.col("Provozovatel_Nazev") + ")")['merged'].to_list()
    counts = (top_stanice['count'] / timeframe).to_list()
    title = f'Největší stanice podle průměrného počtu provedených kontrol za rok'
    path = 'kod/explorace/prohlidky_grafy/prohlidky_top_stanice.svg'

    horizontal_bar(labels, counts, title, path)
    print(f'Podil prohlidek nejvetsi stanice na celku: {top_stanice['count'].item(1) / df.height * 100:.2f} %')

## Prohlidka stanice kraj

In [74]:
if PRINT:
    # Definice převodníku
    kraje_mapping = {
        "Hlavní město Praha": "CZ0100000000",
        "Středočeský kraj": "CZ0200000000",
        "Jihočeský kraj": "CZ0310000000",
        "Plzeňský kraj": "CZ0320000000",
        "Karlovarský kraj": "CZ0410000000",
        "Ústecký kraj": "CZ0420000000",
        "Liberecký kraj": "CZ0510000000",
        "Královéhradecký kraj": "CZ0520000000",
        "Pardubický kraj": "CZ0530000000",
        "Kraj Vysočina": "CZ0630000000",
        "Jihomoravský kraj": "CZ0640000000",
        "Olomoucký kraj": "CZ0710000000",
        "Zlínský kraj": "CZ0720000000",
        "Moravskoslezský kraj": "CZ0800000000"
    }

    # Agregace v Polars
    counts_pl = df.with_columns(pl.col("Prohlidka_Stanice_Kraj").replace(kraje_mapping).alias("kod_kraje")).group_by("kod_kraje").agg((pl.len() / timeframe).alias("pocet"))

    plot_czech_regional_map(counts_pl=counts_pl, value_column="pocet", title='Průměrný roční počet provedených emisních kontrol v jednotlivých krajích', legend_label="Počet kontrol", output_path="kod/explorace/prohlidky_grafy/prohlidky_pocty_mapa.svg")

## Datum prohlidky

In [75]:
if PRINT:
    time_series_all(df['DatumProhlidky'], 'Počty měsíčních měření emisí', 'Počet prohlídek', '1mo', 'kod/explorace/prohlidky_grafy/prohlidky_mesicne.svg')
    time_series_all(df['DatumProhlidky'], 'Počty ročních měření emisí', 'Počet prohlídek', '1y', 'kod/explorace/prohlidky_grafy/prohlidky_rocne.svg')
    time_series_year(df['DatumProhlidky'], 'Průměrné počty měsíčních měření emisí', 'Počet prohlídek', 'kod/explorace/prohlidky_grafy/prohlidky_v_roce.svg')

## Prohlidka zahajeni, Prohlidka ukonceni

In [76]:
if PRINT:
    print(f'Pocet prohlidek se zapornou delkou: {df.filter(pl.col('Prohlidka_Ukonceni') - pl.col('Prohlidka_Zahajeni') < 0).height}')
    print(f'Pocet prohlidek s nulovou delkou: {df.filter(pl.col('Prohlidka_Ukonceni') - pl.col('Prohlidka_Zahajeni') == 0).height}')
    print(f'Pocet mereni emisi zacinajicich drive, nez zacne prohlidka: {df.filter(pl.col('Emise_Zahajeni') - pl.col('Prohlidka_Zahajeni') < 0).height}')
    print(f'Pocet mereni emisi koncicich pozdeji, nez skonci prohlidka: {df.filter(pl.col('Prohlidka_Ukonceni') - pl.col('Emise_Ukonceni') < 0).height}')

In [77]:
if PRINT:
    active_prohlidky_day_plot(
        df=df,
        start_col='Prohlidka_Zahajeni',
        end_col='Prohlidka_Ukonceni',
        title='Průměrný počet současně probíhajících prohlídek v průběhu dne',
        y_title='Počet aktivních prohlídek',
        interval="15m",
        save_path='kod/explorace/prohlidky_grafy/prohlidky_aktivni_ve_dni.svg'
    )

In [78]:
if PRINT:
    # Volání funkce
    duration_prohlidky_plot(
        df=df.filter(pl.col('TechnickaCast_Pritomno') == False),
        start_col='Prohlidka_Zahajeni',
        end_col='Prohlidka_Ukonceni',
        title='Průměrná délka trvání prohlídky dle hodiny zahájení',
        y_title='Doba trvání',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_trvani_podle_zahajeni.svg'
    )

In [79]:
if PRINT:
    # Volání pro lineární měřítko
    duration_density_plot(
        df=df.filter(pl.col('TechnickaCast_Pritomno') == False).filter(pl.col('Prohlidka_Ukonceni') - pl.col('Prohlidka_Zahajeni') < pl.duration(minutes=100)),
        start_col='Prohlidka_Zahajeni',
        end_col='Prohlidka_Ukonceni',
        title='Rozdělení délek trvání prohlídek',
        x_label='Doba trvání v minutách',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_doba_trvani.svg'
    )

    # Volání pro logaritmické měřítko
    duration_density_plot(
        df=df.filter(pl.col('TechnickaCast_Pritomno') == False).filter((pl.col('Prohlidka_Ukonceni') - pl.col('Prohlidka_Zahajeni')).is_between(pl.duration(minutes=1.5), pl.duration(minutes=200))),
        start_col='Prohlidka_Zahajeni',
        end_col='Prohlidka_Ukonceni',
        title='Rozdělení délek trvání prohlídek (Log)',
        x_label='Doba trvání v minutách',
        log_scale=True,
        save_path='kod/explorace/prohlidky_grafy/prohlidky_doba_trvani_log.svg'
    )

## Vozidlo druh

In [82]:
if PRINT:
    vehicle_types = df['Vozidlo_Druh'].value_counts(sort=True)

    horizontal_bar(
        labels=vehicle_types['Vozidlo_Druh'],
        counts=vehicle_types['count'] / df.height,
        title='Hodnoty v příznaku "Vozidlo_Druh"',
        max_bars=3,
        height=4,
        decimals=3,
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_druh.svg'
    )

## Vozidlo provedeni

In [83]:
if PRINT:
    vehicle_style = df['Vozidlo_Provedeni'].fill_null('Chybí').value_counts(sort=True)

    horizontal_bar(
        labels=vehicle_style['Vozidlo_Provedeni'],
        counts=vehicle_style['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Vozidlo_Provedeni"',
        decimals=3,
        max_bars=10,
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_provedeni.svg'
    )

## Vozidlo znacka

In [84]:
if PRINT:
    vehicle_brand = df['Vozidlo_Znacka'].replace('VW', 'VOLKSWAGEN').fill_null('Chybí').value_counts(sort=True)

    horizontal_bar(
        labels=vehicle_brand['Vozidlo_Znacka'],
        counts=vehicle_brand['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Vozidlo_Znacka"',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_znacka.svg',
        decimals=3,
        max_bars=10
    )

## Vozidlo obchodni oznaceni

In [85]:
if PRINT:
    vehicle_name = df['Vozidlo_ObchodniOznaceni'].fill_null('Chybí').value_counts(sort=True)

    horizontal_bar(
        labels=vehicle_name['Vozidlo_ObchodniOznaceni'],
        counts=vehicle_name['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Vozidlo_ObchodniOznaceni"',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_obchodni_oznaceni.svg',
        decimals=3,
        max_bars=10
    )

In [86]:
if PRINT:
    # Pouze prvni slovo
    vehicle_name = df['Vozidlo_ObchodniOznaceni'].str.replace(r"\s+.*$", "").value_counts(sort=True)

    horizontal_bar(
        labels=vehicle_name['Vozidlo_ObchodniOznaceni'],
        counts=vehicle_name['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Vozidlo_ObchodniOznaceni" (po úpravě)',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_provedeni_upraveno.svg',
        decimals=3,
        max_bars=10
    )

## Vozidlo typ motoru

In [87]:
if PRINT:
    engine_type = df['Vozidlo_TypMotoru'].fill_null('Chybí').value_counts(sort=True)

    horizontal_bar(
        labels=engine_type['Vozidlo_TypMotoru'],
        counts=engine_type['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Vozidlo_TypMotoru"',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vozidlo_typ_motoru.svg',
        decimals=3,
        height=4,
        max_bars=4
    )

## Registrace datum prvni

In [88]:
if PRINT:
    registration_after = df.filter(pl.col('Registrace_DatumPrvni') >= pl.col('Prohlidka_Zahajeni'))
    print(f'Pocet vozidel zaregistrovaych az po prohlidce: {len(registration_after)}')
    registration_future = registration_after.filter(pl.col('Registrace_DatumPrvni') >= datetime(2026, 4, 5))
    print(f'Z toho pocet vozidel zaregistrovaych v budoucnosti: {len(registration_future)}')

In [89]:
if PRINT:
    duration_density_plot(
        df=df.filter(pl.col('Prohlidka_Zahajeni') - pl.col('Registrace_DatumPrvni') < pl.duration(days=35*365.24)),
        start_col='Registrace_DatumPrvni',
        end_col='Prohlidka_Zahajeni',
        title='Uplynulá doba od první registrace vozidla při prohlídce',
        x_label='Počet let',
        unit='years',
        log_scale=False,
        save_path='kod/explorace/prohlidky_grafy/prohlidky_doba_od_registrace.svg'
    )

## Registrace stat

In [90]:
if PRINT:
    registration_country = df['Registrace_Stat'].value_counts(sort=True)

    horizontal_bar(
        labels=registration_country['Registrace_Stat'],
        counts=registration_country['count'] / vehicle_style['count'].sum(),
        title='Hodnoty v příznaku "Registrace_Stat"',
        save_path='kod/explorace/prohlidky_grafy/prohlidky_registrace_stat.svg',
        decimals=3,
        max_bars=10
    )

## Emise zakladni palivo

In [91]:
if PRINT:
    basic_fuel = df['Emise_ZakladniPalivo'].value_counts(sort=True)
    print(f'Pomer vozidel jezdicich na benzin: {basic_fuel.filter(pl.col('Emise_ZakladniPalivo') == 'Benzín').item(row=0, column='count') / df.height * 100:.2f} %.')
    print(f'Pomer vozidel jezdicich na naftu: {basic_fuel.filter(pl.col('Emise_ZakladniPalivo') == 'Nafta').item(row=0, column='count') / df.height * 100:.2f} %.')

## Technicka, ADR a TSK cast pritomno

In [92]:
if PRINT:
    cols_to_plot = ['TechnickaCast_Pritomno', 'AdrCast_Pritomno', 'TskCast_Pritomno']

    graph_title = "Podíl záznamů prohlídek obsahující současně informace o jiných úkonech než měření emisí"
    output_file = 'kod/explorace/prohlidky_grafy/prohlidky_pritomne_casti.svg'

    plot_stacked_ratios(df=df, cols=cols_to_plot, title=graph_title, save_path=output_file)

## Vysledek odometr

In [93]:
if PRINT: print(f'Pocet zaznamu s najetymi vice nez 1 000 000 km: {len(df['Vysledek_Odometr'].filter(df['Vysledek_Odometr'] > 1000000))}')

In [94]:
if PRINT:
    distribution_density_plot(
        df=df.filter(pl.col('Vysledek_Odometr') < 600000),
        value_col='Vysledek_Odometr',
        title='Rozložení počtu najetých kilometrů vozidla při měření emisí',
        x_label='Počet kilometrů',
        log_scale=False,
        save_path='kod/explorace/prohlidky_grafy/prohlidky_vysledek_odometr.svg'
    )

## Administrativne opraveno

In [95]:
if PRINT: print(f'Podil administrativne opravenych protokolu: {df['AdministrativneOpraveno'].mean() * 100:.2f} %') # type: ignore

# Uprava casovych hodnot do stavu vhodnejsiho pro dalsi analyzu

In [96]:
# Transformace časových údajů pro detekci anomálií
df = df.with_columns([
    # Výpočet délek trvání v minutách (Duration automaticky řeší rozdíly dnů)
    ((pl.col("Prohlidka_Ukonceni") - pl.col("Prohlidka_Zahajeni")).dt.total_seconds() / 60).alias("Doba_Trvani_Prohlidky"),
    ((pl.col("Emise_Ukonceni") - pl.col("Emise_Zahajeni")).dt.total_seconds() / 60).alias("Doba_Trvani_Emisi"),
    
    # Výpočet stáří vozidla v letech
    ((pl.col("DatumProhlidky") - pl.col("Registrace_DatumPrvni")).dt.total_days() / 365.25).alias("Stari_Vozidla_Let"),
    
    # Výpočet radiantů z času (ms od začátku dne / ms v celém dni * 2pi)
    (((pl.col("Prohlidka_Zahajeni") - pl.col("Prohlidka_Zahajeni").dt.date()).dt.total_milliseconds() / 86_400_000).alias('Prohlidka_Cas_zahajeni'))
]).drop([
    # Odstranění původních sloupců a pomocné proměnné
    "DatumProhlidky", 
    "Prohlidka_Zahajeni", 
    "Prohlidka_Ukonceni", 
    "Emise_Zahajeni", 
    "Emise_Ukonceni", 
    "Registrace_DatumPrvni", 
])

# Ulozeni vysledku

In [100]:
df.write_parquet(r"E:\CVUT_BAP\kod\data\processed\prohlidky_model.parquet")

In [101]:
PRINT = True
if PRINT: describe(df)

CisloProtokolu,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Vozidlo_Druh,Vozidlo_Provedeni,Vozidlo_Znacka,Vozidlo_ObchodniOznaceni,Vozidlo_TypMotoru,Registrace_Stat,Emise_OdpovednaOsoba,Emise_ZakladniPalivo,TechnickaCast_Pritomno,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Prohlidka_Cas_zahajeni
str,i32,i32,str,str,str,str,str,str,str,str,str,i32,str,bool,bool,bool,i32,bool,f64,f64,f64,f64
"""CZ-570505-21-10-0042""",50925,570505,"""Zlínský kraj""","""Zlín""","""Zlín""","""OSOBNÍ AUTOMOBIL""",null,"""OPEL""","""ZAFIRA (T98 MONOCAB)""","""Z18XE""","""Česká republika""",50925,"""Benzín""",false,false,false,222592,false,12.116667,10.85,18.130048,0.569158


(16218646, 23)


,CisloProtokolu,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Vozidlo_Druh,Vozidlo_Provedeni,Vozidlo_Znacka,Vozidlo_ObchodniOznaceni,...,Emise_ZakladniPalivo,TechnickaCast_Pritomno,AdrCast_Pritomno,TskCast_Pritomno,Vysledek_Odometr,AdministrativneOpraveno,Doba_Trvani_Prohlidky,Doba_Trvani_Emisi,Stari_Vozidla_Let,Prohlidka_Cas_zahajeni
0,CZ-570505-21-10-0042,50925,570505,Zlínský kraj,Zlín,Zlín,OSOBNÍ AUTOMOBIL,None,OPEL,ZAFIRA (T98 MONOCAB),...,Benzín,False,False,False,222592,False,12.116667,10.850000,18.130048,0.569158
1,CZ-420409-19-10-0394,43649,420409,Středočeský kraj,Kolín,Kolín,OSOBNÍ AUTOMOBIL,None,MERCEDES-BENZ,C 200,...,Nafta,False,False,False,138448,False,51.483333,14.783333,5.664613,0.331175
2,CZ-460318-24-03-0027,35137,460318,Pardubický kraj,Hlinsko,Hlinsko,OSOBNÍ AUTOMOBIL,AA SEDAN,ŠKODA,OCTAVIA (1U),...,Benzín,False,False,False,192690,False,38.116667,28.166667,19.739904,0.554890
3,CZ-440326-22-06-0936,87301,440326,Karlovarský kraj,Karlovy Vary,Jenišov,OSOBNÍ AUTOMOBIL,AA SEDAN,BMW,540,...,Benzín,False,False,False,49413,False,9.666667,8.950000,3.986311,0.623663
4,CZ-470802-24-02-0314,36965,470802,Zlínský kraj,Kroměříž,Kroměříž,OSOBNÍ AUTOMOBIL,AC KOMBI,ŠKODA,OCTAVIA,...,Benzín,False,False,False,62908,False,11.250000,10.250000,6.061602,0.401237
5,CZ-480562-22-05-1736,72231,480562,Olomoucký kraj,Olomouc,Olomouc,OSOBNÍ AUTOMOBIL,None,ŠKODA,SUPERB (3T),...,Nafta,False,False,False,271696,False,21.033333,17.833333,12.052019,0.380126
6,CZ-471117-20-10-0359,49409,471117,Zlínský kraj,Uherský Brod,Vlčnov,OSOBNÍ AUTOMOBIL,None,ŠKODA,OCTAVIA (1Z),...,Nafta,False,False,False,355263,False,115.866667,10.383333,8.862423,0.318555
7,CZ-470540-22-10-0011,39659,470540,Zlínský kraj,Zlín,Kašava,OSOBNÍ AUTOMOBIL,AC KOMBI,ŠKODA,OCTAVIA COMBI (1U),...,Nafta,False,False,False,379267,False,12.283333,10.033333,17.845311,0.327700
8,CZ-470299-22-10-0787,3189,470299,Jihomoravský kraj,Brno,Brno,OSOBNÍ AUTOMOBIL,None,VOLVO,V 70 (S),...,Nafta,False,False,False,318026,False,275.516667,15.733333,19.082820,0.351250
9,CZ-550309-20-12-0206,2545,550309,Ústecký kraj,Chomutov,Spořice,OSOBNÍ AUTOMOBIL,None,ŠKODA,OCTAVIA COMBI (1U),...,Nafta,False,False,False,295281,False,10.500000,6.516667,16.881588,0.574090


,column,majority_class,majority_cnt,null,dtype
0,CisloProtokolu,CZ-510515-23-06-0714,1 / 16 218 646,0 / 16 218 646,String
1,Prohlidka_OdpovednaOsoba,35165,55 725 / 16 218 646,0 / 16 218 646,Int32
2,Prohlidka_Stanice_Cislo,540514,254 043 / 16 218 646,0 / 16 218 646,Int32
3,Prohlidka_Stanice_Kraj,Středočeský kraj,2 344 213 / 16 218 646,0 / 16 218 646,String
4,Prohlidka_Stanice_ORP,Hlavní město Praha,1 682 833 / 16 218 646,0 / 16 218 646,String
5,Prohlidka_Stanice_Obec,Praha,1 682 833 / 16 218 646,0 / 16 218 646,String
6,Vozidlo_Druh,OSOBNÍ AUTOMOBIL,16 136 468 / 16 218 646,0 / 16 218 646,String
7,Vozidlo_Provedeni,AC KOMBI,4 095 426 / 16 218 646,7 992 325 / 16 218 646,String
8,Vozidlo_Znacka,ŠKODA,5 451 446 / 16 218 646,0 / 16 218 646,String
9,Vozidlo_ObchodniOznaceni,OCTAVIA,1 087 227 / 16 218 646,0 / 16 218 646,String
